# Feature Analysis for Level Generation

Which features best characterize the "style" of original levels?

Goals:
1. **Correlation analysis** — which features are redundant?
2. **PCA** — what's the effective dimensionality?
3. **Bin discrimination** — which features distinguish entity composition bins (especially functional vs decorative gate/traps)?
4. **Feature selection** — pick an informative, low-redundancy feature set for KDE fitting

In [ ]:
import numpy as np
import mummymaze_rust
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

MAZE_DIR = Path("../mazes")
GRID_SIZES = [6, 8, 10]

## 1. Compute all features for every original level

In [ ]:
def wall_count(level):
    d = level.to_dict()
    gs = d["grid_size"]
    h, v = d["h_walls"], d["v_walls"]
    h_interior = sum(h[i] for i in range(gs, gs * gs))
    v_interior = sum(v[r * (gs + 1) + c] for r in range(gs) for c in range(1, gs))
    return h_interior + v_interior


def classify_gate(level, bfs_with):
    d = level.to_dict()
    if not d.get("gate"):
        return "none"
    d_no_gate = {**d, "gate": None, "key": None}
    bfs_without = mummymaze_rust.solve(mummymaze_rust.Level.from_dict(d_no_gate))
    if bfs_without is not None and bfs_without == bfs_with:
        return "decorative"
    gs = d["grid_size"]
    gr, gc = d["gate"]
    v_walls = list(d["v_walls"])
    v_walls[gr * (gs + 1) + gc + 1] = True
    d_wall = {**d, "gate": None, "key": None, "v_walls": v_walls}
    bfs_wall = mummymaze_rust.solve(mummymaze_rust.Level.from_dict(d_wall))
    if bfs_wall is not None and bfs_wall == bfs_with:
        return "passage"
    return "functional"


def classify_traps(level, bfs_with):
    d = level.to_dict()
    if not d.get("traps"):
        return "none"
    d_no = {**d, "traps": []}
    bfs_without = mummymaze_rust.solve(mummymaze_rust.Level.from_dict(d_no))
    if bfs_without is not None and bfs_without == bfs_with:
        return "decorative"
    return "functional"


# All feature names we'll extract
FEATURE_NAMES = [
    "bfs_moves", "n_states", "log_win_prob", "expected_steps",
    "wall_count", "dead_end_ratio", "avg_branching",
    "n_optimal", "greedy_deviation", "path_safety",
]

# Parse, analyze, and classify all levels
print("Parsing and analyzing all levels (this takes a minute)...")
records = {}  # gs -> list of dicts

for gs in GRID_SIZES:
    records[gs] = []

for dat_file in sorted(MAZE_DIR.glob("B-*.dat")):
    try:
        levels = mummymaze_rust.parse_file(str(dat_file))
    except Exception:
        continue
    for sub_idx, lev in enumerate(levels):
        d = lev.to_dict()
        gs = d["grid_size"]
        if gs not in records:
            continue

        a = mummymaze_rust.analyze(lev)
        if a["bfs_moves"] is None:
            continue  # unsolvable

        bfs = a["bfs_moves"]
        log_wp = np.log10(a["win_prob"]) if a["win_prob"] > 0 else -20.0
        gate_cls = classify_gate(lev, bfs)
        trap_cls = classify_traps(lev, bfs)

        has_m2 = d["mummy2"] is not None
        has_sc = d["scorpion"] is not None
        tc = len(d.get("traps", []))

        rec = {
            "level": lev,
            "grid_size": gs,
            "has_mummy2": has_m2,
            "has_scorpion": has_sc,
            "trap_count": tc,
            "gate_class": gate_cls,
            "trap_class": trap_cls,
            # Continuous features
            "bfs_moves": bfs,
            "n_states": a["n_states"],
            "log_win_prob": log_wp,
            "expected_steps": a["expected_steps"],
            "wall_count": wall_count(lev),
            "dead_end_ratio": a["dead_end_ratio"],
            "avg_branching": a["avg_branching_factor"],
            "n_optimal": a["n_optimal_solutions"],
            "greedy_deviation": a["greedy_deviation_count"],
            "path_safety": a["path_safety"] if a["path_safety"] is not None else 0.0,
        }
        records[gs].append(rec)

for gs in GRID_SIZES:
    print(f"  gs={gs}: {len(records[gs])} solvable levels")

# Build feature matrices
feature_matrices = {}
for gs in GRID_SIZES:
    feature_matrices[gs] = np.array([
        [r[f] for f in FEATURE_NAMES] for r in records[gs]
    ])
    print(f"  gs={gs} feature matrix: {feature_matrices[gs].shape}")

## 2. Correlation matrix

In [ ]:
for gs in GRID_SIZES:
    feats = feature_matrices[gs]
    corr = np.corrcoef(feats.T)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(FEATURE_NAMES)))
    ax.set_yticks(range(len(FEATURE_NAMES)))
    ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(FEATURE_NAMES)

    # Annotate cells with correlation values
    for i in range(len(FEATURE_NAMES)):
        for j in range(len(FEATURE_NAMES)):
            color = "white" if abs(corr[i, j]) > 0.6 else "black"
            ax.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center",
                    fontsize=8, color=color)

    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"Feature Correlation Matrix (gs={gs}, n={feats.shape[0]})")
    plt.tight_layout()
    plt.show()

    # Print highly correlated pairs
    print(f"\ngs={gs} — pairs with |r| > 0.7:")
    for i in range(len(FEATURE_NAMES)):
        for j in range(i + 1, len(FEATURE_NAMES)):
            r = corr[i, j]
            if abs(r) > 0.7:
                print(f"  {FEATURE_NAMES[i]:20s} vs {FEATURE_NAMES[j]:20s}: r={r:.3f}")
    print()

## 3. PCA — effective dimensionality

In [ ]:
fig, axes = plt.subplots(1, len(GRID_SIZES), figsize=(6 * len(GRID_SIZES), 5))
if len(GRID_SIZES) == 1:
    axes = [axes]

for ax, gs in zip(axes, GRID_SIZES):
    feats = feature_matrices[gs]
    scaler = StandardScaler()
    normed = scaler.fit_transform(feats)

    pca = PCA()
    pca.fit(normed)

    cumvar = np.cumsum(pca.explained_variance_ratio_)
    ax.bar(range(1, len(cumvar) + 1), pca.explained_variance_ratio_,
           alpha=0.6, label="Individual", color="tab:blue")
    ax.plot(range(1, len(cumvar) + 1), cumvar, "o-",
            color="tab:orange", label="Cumulative")
    ax.axhline(0.95, color="red", linestyle="--", alpha=0.5, label="95%")
    ax.axhline(0.99, color="red", linestyle=":", alpha=0.5, label="99%")
    ax.set_xlabel("Principal Component")
    ax.set_ylabel("Explained Variance Ratio")
    ax.set_title(f"gs={gs}")
    ax.legend(fontsize=8)
    ax.set_xticks(range(1, len(cumvar) + 1))
    ax.grid(True, alpha=0.3)

    # How many components for 95%?
    n95 = int((cumvar >= 0.95).argmax()) + 1
    n99 = int((cumvar >= 0.99).argmax()) + 1
    print(f"gs={gs}: {n95} components for 95%, {n99} for 99%")

    # Show loadings for top 3 components
    print("  Top 3 PC loadings:")
    for pc_idx in range(min(3, len(pca.components_))):
        loadings = pca.components_[pc_idx]
        sorted_idx = np.argsort(np.abs(loadings))[::-1]
        top3 = [(FEATURE_NAMES[i], loadings[i]) for i in sorted_idx[:3]]
        var = pca.explained_variance_ratio_[pc_idx]
        print(f"    PC{pc_idx+1} ({var:.1%}): " +
              ", ".join(f"{n}={v:+.2f}" for n, v in top3))

fig.suptitle("PCA: Explained Variance per Component", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Feature distributions by bin (functional vs decorative)

Do the continuous features separate gate/trap functional vs decorative classes?

In [ ]:
# Gate classification: compare feature distributions across decorative/passage/functional
gs = 8
recs = records[gs]

gate_classes = ["decorative", "passage", "functional"]
gate_colors = {"decorative": "tab:blue", "passage": "tab:green", "functional": "tab:red"}

# Filter to levels that have gates
gate_recs = [r for r in recs if r["gate_class"] != "none"]
print(f"gs={gs}: {len(gate_recs)} levels with gates")
for gc in gate_classes:
    n = sum(1 for r in gate_recs if r["gate_class"] == gc)
    print(f"  {gc}: {n}")

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for i, fname in enumerate(FEATURE_NAMES):
    ax = axes[i // 5, i % 5]
    for gc in gate_classes:
        vals = [r[fname] for r in gate_recs if r["gate_class"] == gc]
        if vals:
            ax.hist(vals, bins=25, alpha=0.5, label=gc, density=True,
                    color=gate_colors[gc])
    ax.set_xlabel(fname)
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Feature Distributions by Gate Class (gs={gs})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Trap classification: compare feature distributions across decorative/functional
trap_classes = ["decorative", "functional"]
trap_colors = {"decorative": "tab:blue", "functional": "tab:red"}

trap_recs = [r for r in recs if r["trap_class"] != "none"]
print(f"gs={gs}: {len(trap_recs)} levels with traps")
for tc in trap_classes:
    n = sum(1 for r in trap_recs if r["trap_class"] == tc)
    print(f"  {tc}: {n}")

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for i, fname in enumerate(FEATURE_NAMES):
    ax = axes[i // 5, i % 5]
    for tc in trap_classes:
        vals = [r[fname] for r in trap_recs if r["trap_class"] == tc]
        if vals:
            ax.hist(vals, bins=25, alpha=0.5, label=tc, density=True,
                    color=trap_colors[tc])
    ax.set_xlabel(fname)
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Feature Distributions by Trap Class (gs={gs})", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Pairwise scatter plots — colored by entity composition

Visualize clustering structure in 2D projections.

In [ ]:
# Scatter matrix of selected feature pairs, colored by entity composition
gs = 8
recs_gs = records[gs]

# Simple entity composition label
def entity_label(r):
    parts = ["2M" if r["has_mummy2"] else "1M"]
    if r["has_scorpion"]:
        parts.append("S")
    if r["trap_count"] > 0:
        parts.append(f"{r['trap_count']}T")
    if r["gate_class"] != "none":
        parts.append("G")
    return " ".join(parts)

labels = [entity_label(r) for r in recs_gs]
unique_labels = sorted(set(labels))
label_to_idx = {l: i for i, l in enumerate(unique_labels)}
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))

# Pick interesting feature pairs
pairs = [
    ("bfs_moves", "n_states"),
    ("bfs_moves", "log_win_prob"),
    ("wall_count", "avg_branching"),
    ("dead_end_ratio", "path_safety"),
    ("greedy_deviation", "bfs_moves"),
    ("n_optimal", "path_safety"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (fx, fy) in zip(axes.flatten(), pairs):
    for lbl in unique_labels:
        mask = [l == lbl for l in labels]
        xs = [recs_gs[i][fx] for i in range(len(recs_gs)) if mask[i]]
        ys = [recs_gs[i][fy] for i in range(len(recs_gs)) if mask[i]]
        ax.scatter(xs, ys, s=5, alpha=0.4, label=lbl, color=colors[label_to_idx[lbl]])
    ax.set_xlabel(fx)
    ax.set_ylabel(fy)
    ax.grid(True, alpha=0.3)

# Single legend for all subplots
handles, leg_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, leg_labels, loc="center right", fontsize=7,
           bbox_to_anchor=(1.12, 0.5), markerscale=3)
fig.suptitle(f"Feature Pair Scatters by Entity Composition (gs={gs})", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Feature importance for bin classification

Train a simple classifier (random forest) to predict the entity composition bin
from continuous features. Feature importances tell us which metrics actually
distinguish between level types.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

gs = 8
recs_gs = records[gs]
X = feature_matrices[gs]

# Task 1: Predict gate class (among levels with gates)
gate_mask = [r["gate_class"] != "none" for r in recs_gs]
X_gate = X[gate_mask]
y_gate = np.array([r["gate_class"] for r in recs_gs if r["gate_class"] != "none"])

rf_gate = RandomForestClassifier(n_estimators=100, random_state=42)
scores_gate = cross_val_score(rf_gate, X_gate, y_gate, cv=5, scoring="accuracy")
print("Gate classification (decorative/passage/functional):")
print(f"  Accuracy: {scores_gate.mean():.3f} +/- {scores_gate.std():.3f}")
print(f"  (baseline = {max(np.unique(y_gate, return_counts=True)[1]) / len(y_gate):.3f})")

rf_gate.fit(X_gate, y_gate)
importances_gate = rf_gate.feature_importances_

# Task 2: Predict trap class (among levels with traps)
trap_mask = [r["trap_class"] != "none" for r in recs_gs]
X_trap = X[trap_mask]
y_trap = np.array([r["trap_class"] for r in recs_gs if r["trap_class"] != "none"])

rf_trap = RandomForestClassifier(n_estimators=100, random_state=42)
scores_trap = cross_val_score(rf_trap, X_trap, y_trap, cv=5, scoring="accuracy")
print("\nTrap classification (decorative/functional):")
print(f"  Accuracy: {scores_trap.mean():.3f} +/- {scores_trap.std():.3f}")
print(f"  (baseline = {max(np.unique(y_trap, return_counts=True)[1]) / len(y_trap):.3f})")

rf_trap.fit(X_trap, y_trap)
importances_trap = rf_trap.feature_importances_

# Task 3: Predict entity composition (simplified)
y_entity = np.array([entity_label(r) for r in recs_gs])
rf_entity = RandomForestClassifier(n_estimators=100, random_state=42)
scores_entity = cross_val_score(rf_entity, X, y_entity, cv=5, scoring="accuracy")
print("\nEntity composition classification:")
print(f"  Accuracy: {scores_entity.mean():.3f} +/- {scores_entity.std():.3f}")
print(f"  ({len(np.unique(y_entity))} classes)")

rf_entity.fit(X, y_entity)
importances_entity = rf_entity.feature_importances_

# Plot importances
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, imp, title in zip(axes,
    [importances_gate, importances_trap, importances_entity],
    ["Gate class", "Trap class", "Entity composition"]):
    sorted_idx = np.argsort(imp)[::-1]
    ax.barh(range(len(FEATURE_NAMES)),
            imp[sorted_idx], color="tab:blue", alpha=0.7)
    ax.set_yticks(range(len(FEATURE_NAMES)))
    ax.set_yticklabels([FEATURE_NAMES[i] for i in sorted_idx])
    ax.set_xlabel("Feature Importance")
    ax.set_title(title)
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Random Forest Feature Importances (gs={gs})", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary: feature redundancy and recommendations

In [ ]:
# Compute cost-adjusted feature rankings
# "Cost" = relative computation time. BFS/states/win_prob come free from ga_evaluate_batch.
# analyze() is needed for the difficulty metrics (slower).

feature_cost = {
    "bfs_moves": 1,       # free from ga_evaluate_batch
    "n_states": 1,        # free
    "log_win_prob": 1,    # free
    "expected_steps": 2,  # needs analyze()
    "wall_count": 0,      # trivial Python computation
    "dead_end_ratio": 2,  # needs analyze()
    "avg_branching": 2,   # needs analyze()
    "n_optimal": 2,       # needs analyze()
    "greedy_deviation": 2,# needs analyze()
    "path_safety": 2,     # needs analyze()
}

gs = 8
feats = feature_matrices[gs]
corr = np.corrcoef(feats.T)

print("=" * 70)
print("FEATURE SUMMARY (gs=8)")
print("=" * 70)

print("\nCorrelation clusters (|r| > 0.7):")
visited = set()
for i in range(len(FEATURE_NAMES)):
    if i in visited:
        continue
    cluster = [i]
    for j in range(i + 1, len(FEATURE_NAMES)):
        if abs(corr[i, j]) > 0.7:
            cluster.append(j)
            visited.add(j)
    if len(cluster) > 1:
        names = [FEATURE_NAMES[k] for k in cluster]
        costs = [feature_cost[FEATURE_NAMES[k]] for k in cluster]
        print(f"  {names} — cheapest: {names[np.argmin(costs)]}")

print("\nAll features ranked by avg RF importance (across 3 classification tasks):")
avg_imp = (importances_gate + importances_trap + importances_entity) / 3
for idx in np.argsort(avg_imp)[::-1]:
    cost = feature_cost[FEATURE_NAMES[idx]]
    print(f"  {FEATURE_NAMES[idx]:20s}: importance={avg_imp[idx]:.3f}  cost={cost}")

print("\nRecommendation: review the correlation matrix and importance plots above")
print("to select a feature set that is informative, low-redundancy, and cheap to compute.")